# Create Welch Foundation Awards (PRIZE PATTERN, GraphQL via Nuxt SPA discovery)

Ingest of **both** of Welch Foundation's annual award programs:
- **Welch Award in Chemistry** — $500,000 USD/year, since 1972 (61 laureates including Robert S. Langer 2026, Peter G. Schultz 2024, Stuart L. Schreiber 2023, JoAnne Stubbe 2017, K.C. Nicolaou 2013, etc.)
- **Norman Hackerman Award in Chemical Research** — $100,000 USD/year (27 laureates including Sheel Dodani 2026, Haotian Wang 2025, etc.; honors young Texas-based chemists)

Source: `welch1.org` — a Nuxt.js + Craft CMS site that exposes its data via a **public GraphQL endpoint** at `https://welch1.org/api` (no authentication). Method #2 on the runbook ladder (REST/GraphQL API), discovered via SPA state inspection — **first ingest on the project to use this discovery playbook**:

1. Loaded `/awards/welch-award-in-chemistry/past-recipients` in the browser.
2. The page is a Nuxt SPA with state embedded as `window.__NUXT__ = (function(...){return {...}}(...))`.
3. The embedded payload showed `awardRecipient` entry references but no inline laureate list — the data comes from server fetches via Apollo.
4. The page's Apollo cache (in `__NUXT__.apollo.defaultClient`) revealed the underlying Craft GraphQL schema.
5. Probing common paths found `/api` returns `{"errors":[{"message":"No GraphQL query was supplied"}]}` — a public GraphQL endpoint with NO auth.
6. GraphQL introspection confirmed the `awardRecipients_awardRecipient_Entry` type with `awardYear`, `awardCategory`, `recipientAffiliation`, `recipientDescription`, etc.

This is distinct from prior methods: REST APIs (CIFAR/Hertz), FacetWP (Hewlett), CKAN/Socrata (CONAHCYT), static-HTML (Holberg/Blue Planet/Templeton/Stockholm), and bulk-CSV (NEH). Documented in the script header so future contributors can apply the same playbook to other Nuxt+Apollo+Craft sites.

**Awarding body:** Welch Foundation — `F4320306196` (US, DOI `10.13039/100000928`).

**Corpus** (verified 2026-05-22 via single GraphQL call): **88 recipient rows** spanning **1972-2026**.
- Welch Award in Chemistry: 61 rows
- Norman Hackerman Award in Chemical Research: 27 rows
- **100% coverage** on name, year, affiliation, description, amount — the GraphQL response carries every field cleanly (richer than what HTML scraping would have produced).

**Schema:**
- `funder_award_id` = `welch-{category-slug}-{year}-{slug}` (e.g., `welch-welch-award-in-chemistry-1972-foo-bar`). Includes the category in the ID so the two award programs can't collide. Verified unique across the 88-row corpus.
- `funder_scheme` distinguishes the two programs: `'Welch Award in Chemistry'` vs `'Norman Hackerman Award in Chemical Research'`. **First multi-scheme PRIZE ingest** on the project — analogous to Crafoord (multi-category) but with distinct *amount tiers* per scheme.
- `amount` is conditional on scheme: `$500,000` for Welch Award, `$100,000` for Hackerman Award. Both amounts verified from each award's own overview page on welch1.org.
- `lead_investigator` PERSON-LEVEL with `given_name` + `family_name` from `split_name()` (runbook §2.4.1). `affiliation.name` populated from `recipientAffiliation` (100% coverage — every recipient has their home institution).
- `funding_type = 'prize'`.
- `declined` always FALSE.

**Source authority** — `welch1.org` is Welch Foundation's own site (Nuxt frontend + Craft CMS backend; the GraphQL endpoint is the public-facing query interface). Method #2 (REST/GraphQL API) on the runbook ladder.

**Prerequisites**: Run `scripts/local/welch_award_to_s3.py` first to make a single GraphQL POST to `https://welch1.org/api` (~5 sec wall-clock), parse 88 recipients, write parquet, upload to S3.

**Data source**: `https://welch1.org/api` (GraphQL POST; query in script header)

**S3 location**: `s3a://openalex-ingest/awards/welch/welch_awards.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.welch_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/welch/welch_awards.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.welch_raw;

## Step 1.5: Inspect raw + multi-scheme amount-by-category sanity check

All source columns from the GraphQL extraction. **Verified per-row coverage on the full extraction** (88 recipients, validated 2026-05-22):
- 100% `funder_award_id`, `year`, `slug`, `name`, `landing_page_url`
- 100% `amount` — variable per scheme ($500K Welch / $100K Hackerman)
- 100% `affiliation`, `description`
- 1972-2026 year range
- 88 rows = 61 Welch Award + 27 Hackerman Award

Source columns: `funder_award_id`, `year`, `slug`, `name`, `given_name`, `family_name`, `category_title`, `category_slug`, `scheme`, `affiliation`, `blurb`, `description`, `display_name`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `is_past_recipient`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.welch_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.welch_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirms the multi-scheme amount tiers.
-- Expected: distinct (scheme, amount) pairs: ('Welch Award in Chemistry', 500000)
-- and ('Norman Hackerman Award in Chemical Research', 100000).
SELECT scheme,
       MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
       MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
       AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
       COUNT(*) AS rows
FROM openalex.awards.welch_raw
GROUP BY scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution
SELECT TRY_CAST(year AS INT) AS year_int,
       scheme,
       COUNT(*) AS rows
FROM openalex.awards.welch_raw
GROUP BY year_int, scheme
ORDER BY year_int DESC;

## Step 1.6: Fail-fast — verify Welch Foundation funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306196;

## Step 2: Transform to award schema (multi-scheme funder)

Each row preserves its category in `funder_scheme` and ships the category-appropriate amount. The `provenance` is the same (`'welch_foundation'`) since both programs are run by the same body — distinct `funder_scheme` values are how downstream queries differentiate.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.welch_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306196  -- Welch Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,  -- variable per scheme; verified in source script
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.scheme AS funder_scheme,  -- 'Welch Award in Chemistry' OR 'Norman Hackerman Award in Chemical Research'
    'welch_foundation' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator: PERSON-LEVEL. Affiliation is 100% populated
    -- from the GraphQL `recipientAffiliation` field.
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.affiliation AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.welch_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 102

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'welch_foundation' AND priority = 102;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    102 AS priority  -- Welch Foundation priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.welch_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (variable per scheme). Verified expectations from the 2026-05-22 extraction:
- Row count: **88** (61 Welch + 27 Hackerman)
- 100% `pct_amount` ($500K Welch, $100K Hackerman)
- `currencies = [USD]`
- 2 distinct `funder_scheme` values (Welch Award in Chemistry, Norman Hackerman Award in Chemical Research), 1 distinct `funding_type` ('prize')
- 51+ distinct years (1972-2026; Hackerman started later than Welch)
- 100% `affiliation` populated
- `declined = FALSE` everywhere

In [ ]:
%sql
SELECT COUNT(*) AS total_welch_award_rows FROM openalex.awards.welch_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.welch_awards;

In [ ]:
%sql
-- §6.3 Data completeness — expect 100% across all fields.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.welch_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
-- Expected: 2 distinct (scheme, amount) tiers; min=100K, max=500K, avg between.
SELECT
    funder_scheme,
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount
FROM openalex.awards.welch_awards
GROUP BY funder_scheme
ORDER BY funder_scheme;

In [ ]:
%sql
-- Year coverage per scheme
SELECT funder_scheme,
       MIN(start_year) AS first_year,
       MAX(start_year) AS last_year,
       COUNT(DISTINCT start_year) AS distinct_years,
       COUNT(*) AS rows
FROM openalex.awards.welch_awards
GROUP BY funder_scheme;

In [ ]:
%sql
-- Sample notable recent + earliest
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS affiliation,
       amount, funder_scheme, start_year
FROM openalex.awards.welch_awards
WHERE start_year >= 2024 OR start_year <= 1975
ORDER BY funder_scheme, start_year DESC
LIMIT 15;

In [ ]:
%sql
-- Top affiliations across both programs
SELECT lead_investigator.affiliation.name AS affiliation,
       COUNT(*) AS recipient_count
FROM openalex.awards.welch_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY affiliation
ORDER BY recipient_count DESC
LIMIT 10;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.welch_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined must be FALSE
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.welch_awards
GROUP BY declined;